<a href="https://colab.research.google.com/github/PIER-202621036/proyecto-gestion-vacaciones/blob/main/notebooks/03_criticidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==========================================
# ZONA COMUN DE datos.py: depuracion de datos
# ==========================================

"""Datos del proyecto: carga, limpieza y acceso a las hojas."""

import pandas as pd
from google.colab import files

# Subir archivo Excel desde la computadora
archivos = files.upload()

# Obtener el nombre del archivo subido
URL_EXCEL = list(archivos.keys())[0]

# Leer archivo Excel
datos = pd.read_excel(URL_EXCEL)

# Mostrar los primeros registros
print(datos.head())

# Hojas que usa Python (las demas del Excel son material de gestion)
HOJAS = [
    "Empleados",                    #0
    "Puestos",                      #1
    "Vacaciones",                   #2
    "Programacion_Vacaciones",      #3
    "Ofertas_Compensacion",         #4
    "Parametros_Vacaciones",        #5
    "Parametros_Criticidad",        #6
    "Niveles_Criticidad"            #7
]

# Unicas columnas que pueden estar vacias (segun el Diccionario)
NULOS_PERMITIDOS = ["motivo", "fecha_estado"]

def cargar_parametros(ruta=URL_EXCEL):
  """Lee la hoja Parametros_Vacaciones y devuelve {regla: valor}."""
  try:
      df = pd.read_excel(ruta, sheet_name="Parametros_Vacaciones")
  except (OSError, ValueError) as error:
      raise ConnectionError(
          f"No se pudo leer Parametros_Vacaciones: {error}"
      ) from error

  parametros = {}
  for fila in df.to_dict("records"):
      parametros[fila["regla"]] = int(fila["valor"])
  return parametros


# Reglas del negocio: se leen de la hoja Parametros_Vacaciones del Excel
PARAMETROS = cargar_parametros()
DIAS_VACACIONES_PERIODO = PARAMETROS["dias_vacaciones_periodo"]
PORCENTAJE_VENTA = PARAMETROS["porcentaje_venta"]
PERIODOS_CRITICOS = PARAMETROS["periodos_criticos"]
PORCENTAJE_DESCUENTO = PARAMETROS["porcentaje_descuento"]
DIAS_ALERTA_VENCIMIENTO = PARAMETROS["dias_alerta_vencimiento"]

# La fecha de corte no esta en la hoja: es la fecha de los datos
FECHA_CORTE = "2026-10-12"

# Textos que usan las hojas (se escriben una sola vez aqui)
ESTADO_GOZADA = "Gozada"
ESTADO_PROGRAMADA = "Programada"
DECISION_APROBADA = "Aprobada"
DECISION_PENDIENTE = "Pendiente"
TIPO_VENTA = "Venta_50"
RESPUESTA_ACEPTO = "Acepto" # A MEJORAR EN EL EXCEL NADA DE TILDES NI EXCEL NI PY

TIPO_INDEMNIZACION = "Indemnizacion"
OFERTA_PAGADA = "Pagado"
OFERTA_AVISADA = "Avisado"



# Cargar todas las hojas del Excel
def cargar_datos(ruta=URL_EXCEL):
  """
    Lee el archivo Excel completo, valida que contenga las hojas obligatorias
    y devuelve un diccionario {nombre_hoja: dataframe}.
  """
  try:
    df = pd.read_excel(ruta, sheet_name=None)
  except OSError as error:
    raise ConnectionError(f"No se pudo leer el Excel: {error}") from error

  datos = {}
  for nombre_hoja in HOJAS:
    if nombre_hoja not in df:
        raise ValueError(f"El Excel no tiene la hoja {nombre_hoja}")

    # Guardar la hoja tal cual está
    datos[nombre_hoja] = df[nombre_hoja]
  return datos

# Limpiar datos
def limpiar_datos(data_excel):
  # Se calcula una sola vez, no en cada columna
  permitidos = [n.lower() for n in NULOS_PERMITIDOS]

  data_limpia = {} # diccionario NUEVO

  for nombre_hoja, df in data_excel.items():

    # 1. Duplicados
    cantidad_duplicados = df.duplicated().sum()
    if cantidad_duplicados > 0:
        print(f"Hoja {nombre_hoja}: {cantidad_duplicados} duplicados.")
        df = df.drop_duplicates()

    # 2. Nulos (excluyendo NULOS_PERMITIDOS), Comprensión de Listas
    columnas_obligatorias = [
        col
        for col in df.columns if col.lower() not in permitidos
    ]

    # Cuenta las CELDAS vacias en las columnas obligatorias:
    # isna() marca True/False y el primer sum() suma por columna;
    # el segundo sum() suma todas las columnas en un solo numero.
    cantidad_nulos = df[columnas_obligatorias].isna().sum().sum()

    if cantidad_nulos > 0:
        print(f"Hoja {nombre_hoja}: {cantidad_nulos} nulos no permitidos.")
        df = df.dropna(subset=columnas_obligatorias)

    # Guardamos la hoja limpia en el diccionario nuevo
    data_limpia[nombre_hoja] = df.reset_index(drop=True)

  return data_limpia

  # ==================================================================
# CONSULTA DE PUESTOS
# ==================================================================

# 1. Consultar los datos de un puesto
puesto_id = "P09"

resultado_puesto = datos.obtener_puesto(puesto_id, vacaciones_excel_depurado)

print("\n\nMostrando detalles del Puesto:", puesto_id)
print("=" * 80)

# Mostrar los datos del puesto
for dato in resultado_puesto:
    print(dato, ":", resultado_puesto[dato])


# ==================================================================
# CONSULTA DE REGLAS DE CRITICIDAD
# ==================================================================

# 2. Consultar una variable específica
nombre_variable = "importancia"

resultado_regla = datos.obtener_regla(nombre_variable, vacaciones_excel_depurado)


# 3. Mostrar todas las variables de criticidad
print("\n\nMostrando las variables de criticidad")
print("=" * 80)

tabla_criticidad = vacaciones_excel_depurado["Parametros_Criticidad"]

for nombre in tabla_criticidad["variable"]:

    regla = datos.obtener_regla(nombre, vacaciones_excel_depurado)

    print(
        nombre,
        "- peso:", regla["peso"],
        "- notas:", regla["nota_min"], "a", regla["nota_max"]
    )


# 4. Mostrar los detalles de una variable específica
print("\n\nMostrando la regla de la variable:", nombre_variable)
print("=" * 80)

for dato in resultado_regla:
    print(dato, ":", resultado_regla[dato])


# ==================================================================
# CONTROL DE ERRORES
# ==================================================================

# 5. Comprobar qué sucede cuando la variable no existe
print("\n\nMostrando control de errores")
print("=" * 80)

try:
    datos.obtener_regla("xyz", vacaciones_excel_depurado)

except ValueError as error:
    print("Error controlado:", error)


# ==================================================================
# NIVELES DE CRITICIDAD
# ==================================================================

# 6. Consultar los niveles de criticidad
resultado_niveles = datos.obtener_niveles(vacaciones_excel_depurado)

print("\n\nMostrando los niveles de criticidad:")
print("=" * 80)

print(pd.DataFrame(resultado_niveles))


# ==================================================================
# CALCULO DE CRITICIDAD
# ==================================================================

# 7. Calcular el score del puesto
score = criticidad.calcular_score(puesto_id, vacaciones_excel_depurado)

print("\n\nMostrando el score del puesto:", puesto_id)
print("=" * 80)

print(score)

KeyboardInterrupt: 